# Aarya Sequential Faults — Step-by-Step Quantitative Extraction

Walks through every internal module call in the metrics extraction pipeline for one fault bucket at a time.

**Pipeline steps exposed:**
1. `load_trace_file` — load spans + bucket metadata
2. `_create_batches` — split spans into batches of 6
3. `_extract_batch_quantitative` — per-batch LLM extraction (structured output)
4. `_identify_detection_mitigation_spans` — full-span LLM scan for detection/mitigation
5. `_validate_bucket_timestamps_with_llm` — validate `bucket.detected_at` (skips if null)
6. `QuantitativeAggregator.aggregate` — code-only numeric aggregation
7. LLM text consolidation — merge descriptive text fields across batches
8. Final override — code values overwrite LLM numbers → `LLMQuantitativeExtraction`

## 1. Imports & Path Setup

In [ ]:
import sys, os, json, asyncio, logging
from pathlib import Path
from pprint import pprint

CERTIFIER_ROOT = Path.cwd()
while CERTIFIER_ROOT.name != "certifier" and CERTIFIER_ROOT != CERTIFIER_ROOT.parent:
    CERTIFIER_ROOT = CERTIFIER_ROOT.parent
if str(CERTIFIER_ROOT) not in sys.path:
    sys.path.insert(0, str(CERTIFIER_ROOT))

print(f"Root: {CERTIFIER_ROOT}")

from metrics_extractor.scripts.metrics_extractor_from_trace import (
    TraceMetricsExtractor, PROMPTS, MODULE_CONFIG,
)
from metrics_extractor.scripts.span_aggregator import QuantitativeAggregator
from metrics_extractor.schema.metrics_model import LLMQuantitativeExtraction

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
print(f"Batch size from config: {MODULE_CONFIG['extractor']['batch_size']}")

## 2. Select Fault Bucket

Change `FAULT_NAME` to switch between the three faults.

In [ ]:
FAULT_BUCKETS_DIR = Path(
    r"C:\Users\meemankgupta\Music\Project\infosys\certifier"
    r"\data\output\08-05-26-aarya\1960bc89"
    r"\f6152c39-sequential-defaults"
    r"\f6152c39-27c4-4601-a5e8-bee80da8ec78\fault_buckets"
)

# Options: "pod-cpu-hog" | "pod-memory-hog" | "pod-network-loss"
FAULT_NAME = "pod-cpu-hog"

BUCKET_FILE = FAULT_BUCKETS_DIR / f"raw_trace_sequential_bucket_{FAULT_NAME}.json"
print(f"Bucket file: {BUCKET_FILE}")
print(f"Exists: {BUCKET_FILE.exists()}")

## Step 1 — `load_trace_file`: Load Spans + Bucket Metadata

In [ ]:
extractor = TraceMetricsExtractor()
spans = extractor.load_trace_file(str(BUCKET_FILE))

print(f"Spans loaded: {len(spans)}")
print(f"\nBucket metadata keys: {list(extractor.bucket_metadata.keys())}")
print(f"  fault_id:              {extractor.bucket_metadata.get('fault_id')}")
print(f"  fault_name:            {extractor.bucket_metadata.get('fault_name')}")
print(f"  injection_timestamp:   {extractor.bucket_metadata.get('injection_timestamp')}")
print(f"  injection_end_timestamp:{extractor.bucket_metadata.get('injection_end_timestamp')}")
print(f"  detected_at:           {extractor.bucket_metadata.get('detected_at')}")
print(f"  mitigated_at:          {extractor.bucket_metadata.get('mitigated_at')}")

print(f"\nSpan names:")
for i, s in enumerate(spans, 1):
    print(f"  {i:2d}. [{s.get('type','?')}] {s.get('name','')}  @ {s.get('startTime','')}")

## Step 2 — `_create_batches`: Split into Batches of 6

In [ ]:
batches = extractor._create_batches(spans)
print(f"Total batches: {len(batches)}  (batch_size={extractor.BATCH_SIZE})")
for i, batch in enumerate(batches, 1):
    names = [s.get('name', '?') for s in batch]
    print(f"  Batch {i} ({len(batch)} spans): {names}")

## Step 3 — `_extract_batch_quantitative`: Per-Batch LLM Extraction

Each batch is sent to the LLM with the quantitative prompt (structured output → `LLMQuantitativeExtraction`).
Results are raw partial metrics — no aggregation yet.

In [ ]:
extractor._init_llm_client()
total_batches = len(batches)
partial_metrics = []

for i, batch in enumerate(batches, 1):
    print(f"\n--- Batch {i}/{total_batches} ({len(batch)} spans) ---")
    result = await extractor._extract_batch_quantitative(batch, i, total_batches)
    partial_metrics.append(result)

    # Show key fields extracted from this batch
    print(f"  detection_success:          {result.get('detection_success')}")
    print(f"  fault_detected:             {result.get('fault_detected')}")
    print(f"  detected_fault_type:        {result.get('detected_fault_type')}")
    print(f"  agent_fault_detection_time: {result.get('agent_fault_detection_time')}")
    print(f"  agent_fault_mitigation_time:{result.get('agent_fault_mitigation_time')}")
    print(f"  input_tokens:               {result.get('input_tokens')}")
    print(f"  output_tokens:              {result.get('output_tokens')}")
    print(f"  tool_calls count:           {len(result.get('tool_calls', []))}")
    print(f"  pii_detection:              {result.get('pii_detection')}")

print(f"\nAll {total_batches} batches extracted.")

### Partial Metrics — Raw Batch Outputs

In [ ]:
for i, pm in enumerate(partial_metrics, 1):
    print(f"\n=== Partial Metrics — Batch {i} ===")
    print(json.dumps(pm, indent=2, default=str))

## Step 4 — `_identify_detection_mitigation_spans`: Full-Span LLM Scan

Sends **all spans with full input + output** (no truncation) in one LLM call.
Returns the span IDs for the first detection and final mitigation events.

> Works even when `bucket.detected_at` is null (sequential faults).

In [ ]:
print(f"bucket.detected_at = {extractor.bucket_metadata.get('detected_at')}  "
      f"(null → will rely entirely on this LLM call)")

span_times = await extractor._identify_detection_mitigation_spans(spans)

print(f"\nSpan identification result:")
for k, v in span_times.items():
    print(f"  {k}: {v}")

## Step 5 — `_validate_bucket_timestamps_with_llm`: Validate Bucket Timestamps

Only runs if `bucket.detected_at` or `bucket.mitigated_at` are non-null.
For sequential faults both are null → returns `{}` immediately.

In [ ]:
detected_at = extractor.bucket_metadata.get("detected_at")
mitigated_at = extractor.bucket_metadata.get("mitigated_at")

if detected_at or mitigated_at:
    print("Bucket has timestamps — running LLM validation...")
    validated_bucket_timestamps = await extractor._validate_bucket_timestamps_with_llm(spans)
    print(f"Validation result: {validated_bucket_timestamps}")
else:
    validated_bucket_timestamps = {}
    print(f"bucket.detected_at={detected_at}, bucket.mitigated_at={mitigated_at}")
    print("Both null → validation skipped. Will use span_identification result from Step 4.")

## Step 6 — `QuantitativeAggregator.aggregate`: Code-Only Numeric Aggregation

Pure Python — no LLM. Sums tokens, merges tool calls, computes ratios, resolves detection time.
`detection_success` is now derived from the resolved `agent_fault_detection_time` (our fix).

In [ ]:
code_aggregated = extractor.quant_aggregator.aggregate(
    partial_metrics=partial_metrics,
    total_spans=len(spans),
    span_times=span_times,
    bucket_metadata=extractor.bucket_metadata,
    validated_bucket_timestamps=validated_bucket_timestamps or None,
)

print("=== Code-Aggregated Fields ===")
for k, v in code_aggregated.items():
    if k == "tool_calls":
        print(f"  tool_calls:           [{len(v)} entries]")
    else:
        print(f"  {k:<35} {v}")

## Step 7 — LLM Text Consolidation

Merges descriptive text fields (`fault_detected`, `injected_fault_name`, `fault_namespace`, etc.)
across all batch partial metrics. Numeric fields are set to null by the LLM and overridden in Step 8.

In [ ]:
from utils.azure_openai_util import AzureLLMClient

consolidation_message = f"""Consolidate text fields from these partial metrics from {len(partial_metrics)} batches.
ONLY consolidate descriptive/text fields (fault_detected, injected_fault_name, injected_fault_category, detected_fault_type, fault_target_service, fault_namespace, experiment_id).
Do NOT compute any numeric values — all numbers are handled by code.

Partial data from batches:
```json
{json.dumps(partial_metrics, indent=2)}
```

Total spans in trace: {len(spans)}"""

llm_result_raw, token_usage = await extractor.llm_client.with_structured_output(
    model_name="gpt-4o",
    messages=consolidation_message,
    output_format=LLMQuantitativeExtraction,
    max_tokens=1500,
    system_prompt=PROMPTS["quantitative_aggregation"],
)
extractor.token_usage.add(token_usage)

if isinstance(llm_result_raw, LLMQuantitativeExtraction):
    llm_text_result = llm_result_raw
else:
    llm_text_result = LLMQuantitativeExtraction.model_validate(llm_result_raw)

print("=== LLM Text Consolidation Result (text fields only) ===")
print(f"  fault_detected:          {llm_text_result.fault_detected}")
print(f"  injected_fault_name:     {llm_text_result.injected_fault_name}")
print(f"  injected_fault_category: {llm_text_result.injected_fault_category}")
print(f"  detected_fault_type:     {llm_text_result.detected_fault_type}")
print(f"  fault_target_service:    {llm_text_result.fault_target_service}")
print(f"  fault_namespace:         {llm_text_result.fault_namespace}")
print(f"  experiment_id:           {llm_text_result.experiment_id}")

## Step 8 — Final Override: Code Values → Final `LLMQuantitativeExtraction`

All code-computed numeric fields from Step 6 overwrite whatever the LLM produced in Step 7.

In [ ]:
for field_name, value in code_aggregated.items():
    if hasattr(llm_text_result, field_name) and value is not None:
        setattr(llm_text_result, field_name, value)

final_result = llm_text_result

print(f"=== FINAL Quantitative Metrics — {FAULT_NAME} ===")
print(f"  detection_success:           {final_result.detection_success}")
print(f"  fault_detected:              {final_result.fault_detected}")
print(f"  detected_fault_type:         {final_result.detected_fault_type}")
print(f"  fault_target_service:        {final_result.fault_target_service}")
print(f"  fault_namespace:             {final_result.fault_namespace}")
print(f"  fault_injection_time:        {final_result.fault_injection_time}")
print(f"  agent_fault_detection_time:  {final_result.agent_fault_detection_time}")
print(f"  agent_fault_mitigation_time: {final_result.agent_fault_mitigation_time}")
print(f"  time_to_detect (s):          {final_result.time_to_detect}")
print(f"  time_to_mitigate (s):        {final_result.time_to_mitigate}")
print(f"  input_tokens:                {final_result.input_tokens}")
print(f"  output_tokens:               {final_result.output_tokens}")
print(f"  trajectory_steps:            {final_result.trajectory_steps}")
print(f"  tool_calls count:            {len(final_result.tool_calls)}")
print(f"  tool_selection_accuracy:     {final_result.tool_selection_accuracy}")
print(f"  pii_detection:               {final_result.pii_detection}")
print(f"  number_of_pii_instances:     {final_result.number_of_pii_instances_detected}")

## Full JSON Output

In [ ]:
print(final_result.model_dump_json(indent=2))

## Token Usage

In [ ]:
usage = extractor.token_usage.to_dict()
print(f"Input tokens:  {usage['input_tokens']:,}")
print(f"Output tokens: {usage['output_tokens']:,}")
print(f"Total tokens:  {usage['total_tokens']:,}")